# Breast Cancer Diagnostic Predictor
## Phase 2 — Exploratory Data Analysis (EDA)

**Dataset:** Breast Cancer Wisconsin (Diagnostic) — UCI / Kaggle  
**Author:** Muzammil Ansari  
**Program:** Post-Baccalaureate Diploma in Business Analytics, UFV

---

### Objective
Understand the clinical story in the data before modelling. Answer the question:
> *What does a malignant tumour look like in data, and how does it differ from a benign one?*

### Charts Produced
1. Diagnosis distribution (class imbalance)
2. Top 3 predictor box plots (Benign vs Malignant)
3. Correlation heatmap (multicollinearity check)
4. Mean feature comparison (Benign vs Malignant averages)
5. PCA scatter plot (visual separability in 2D)
6. Feature importance bar chart (correlation ranking)

In [ ]:
# Block 1 — Upload clean dataset
from google.colab import files
uploaded = files.upload()

In [ ]:
# Block 2 — Load data and import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('breast_cancer_clean.csv')
print('Shape:', df.shape)
print('Libraries loaded successfully')

In [ ]:
# Chart 1 — Diagnosis Distribution
# Clinical note: 37.3% malignancy rate means accuracy alone is misleading
# A model predicting 'Benign' for everyone scores 62.7% while missing all cancers

ax = sns.countplot(x='diagnosis', data=df, palette=['#2ecc71', '#e74c3c'])

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=13, fontweight='bold')

plt.title('Diagnosis Distribution: Benign vs Malignant', fontsize=16, fontweight='bold')
plt.xlabel('Diagnosis (0 = Benign, 1 = Malignant)', fontsize=13)
plt.ylabel('Number of Patients', fontsize=13)
plt.xticks([0, 1], ['Benign (0)', 'Malignant (1)'], fontsize=12)

plt.tight_layout()
plt.savefig('chart1_diagnosis_distribution.png', dpi=150)
plt.show()
print('Chart 1 saved.')

In [ ]:
# Chart 2 — Top 3 Predictors: Box Plots by Diagnosis
# Finding: Malignant cases show consistently higher concave points and larger perimeters
# Clinical interpretation: malignant cells are larger and more irregular in shape

top_features = ['concave points_worst', 'perimeter_worst', 'concave points_mean']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, feature in enumerate(top_features):
    sns.boxplot(x='diagnosis', y=feature, data=df,
                palette=['#2ecc71', '#e74c3c'], ax=axes[i])
    axes[i].set_title(f'{feature}', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Diagnosis (0=Benign, 1=Malignant)', fontsize=11)
    axes[i].set_ylabel(feature, fontsize=11)
    axes[i].set_xticks([0, 1])
    axes[i].set_xticklabels(['Benign', 'Malignant'])

plt.suptitle('Top 3 Predictors: Benign vs Malignant Distribution',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart2_top3_boxplots.png', dpi=150)
plt.show()
print('Chart 2 saved.')

In [ ]:
# Chart 3 — Correlation Heatmap
# Finding: Strong multicollinearity exists — radius, perimeter, area all measure size differently
# DataRobot's 'Informative Features' selection handles this automatically in Phase 3

corr_matrix = df.corr()

plt.figure(figsize=(20, 16))
sns.heatmap(corr_matrix,
            cmap='RdYlGn',
            center=0,
            linewidths=0.5,
            annot=False,
            square=True)

plt.title('Feature Correlation Heatmap', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig('chart3_correlation_heatmap.png', dpi=150)
plt.show()
print('Chart 3 saved.')

In [ ]:
# Chart 4 — Mean Feature Comparison: Benign vs Malignant
# Finding: area_mean shows the largest gap between Benign and Malignant
# Confirms cell size as a primary diagnostic marker

mean_features = [col for col in df.columns if '_mean' in col]
mean_comparison = df.groupby('diagnosis')[mean_features].mean()

mean_comparison.T.plot(kind='bar', figsize=(16, 7),
                        color=['#2ecc71', '#e74c3c'],
                        edgecolor='white', width=0.7)

plt.title('Average Feature Values: Benign vs Malignant', fontsize=16, fontweight='bold')
plt.xlabel('Features', fontsize=13)
plt.ylabel('Average Value', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.legend(['Benign (0)', 'Malignant (1)'], fontsize=12)
plt.tight_layout()
plt.savefig('chart4_mean_features_comparison.png', dpi=150)
plt.show()
print('Chart 4 saved.')

In [ ]:
# Chart 5 — PCA Scatter Plot
# Finding: 63.3% of variance captured in 2 dimensions
# Benign and Malignant form visually distinct clusters even in reduced space
# Confirms features carry strong discriminative signal for the model

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1],
                      c=y, cmap='RdYlGn',
                      alpha=0.7, edgecolors='white',
                      linewidth=0.5, s=80)

plt.colorbar(scatter, label='Diagnosis (0=Benign, 1=Malignant)')
plt.title('PCA: Can We Visually Separate Benign from Malignant?',
          fontsize=16, fontweight='bold')
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)',
           fontsize=12)
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)',
           fontsize=12)

plt.tight_layout()
plt.savefig('chart5_pca_scatter.png', dpi=150)
plt.show()

print(f'Chart 5 saved.')
print(f'Total variance explained: {sum(pca.explained_variance_ratio_)*100:.1f}%')

In [ ]:
# Chart 6 — Feature Importance: Correlation Ranking
# Top 5 findings: concave points_worst (0.794), perimeter_worst (0.783),
# concave points_mean (0.777), radius_worst (0.776), perimeter_mean (0.742)
# All measure cell SIZE or SHAPE IRREGULARITY — the two core malignancy markers

correlations = df.corr()['diagnosis'].drop('diagnosis').abs().sort_values(ascending=False)
top15 = correlations.head(15)

plt.figure(figsize=(12, 8))
bars = plt.barh(top15.index[::-1], top15.values[::-1],
                color='#e74c3c', edgecolor='white', height=0.7)

for bar, val in zip(bars, top15.values[::-1]):
    plt.text(val + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=10, fontweight='bold')

plt.title('Top 15 Features Most Correlated with Malignancy',
          fontsize=16, fontweight='bold')
plt.xlabel('Absolute Correlation with Diagnosis', fontsize=13)
plt.ylabel('Feature', fontsize=13)
plt.xlim(0, 1.0)
plt.tight_layout()
plt.savefig('chart6_feature_importance_correlation.png', dpi=150)
plt.show()
print('Chart 6 saved.')

In [ ]:
# Download all charts
from google.colab import files

charts = [
    'chart1_diagnosis_distribution.png',
    'chart2_top3_boxplots.png',
    'chart3_correlation_heatmap.png',
    'chart4_mean_features_comparison.png',
    'chart5_pca_scatter.png',
    'chart6_feature_importance_correlation.png'
]

for chart in charts:
    files.download(chart)
    print(f'Downloaded: {chart}')

---
## Phase 2 Summary — Clinical Findings

| Chart | Finding |
|---|---|
| Diagnosis Distribution | 62.7% Benign, 37.3% Malignant — class imbalance exists |
| Box Plots (Top 3) | Malignant cases show higher concave points and larger perimeters |
| Correlation Heatmap | Strong multicollinearity — many features measure overlapping properties |
| Mean Feature Comparison | `area_mean` shows largest gap between Benign and Malignant |
| PCA Scatter | Two classes form distinct clusters in 2D (63.3% variance captured) |
| Feature Importance | Top 5 predictors all measure cell size or shape irregularity |

### Core Clinical Narrative
> *"Malignant tumours are characterized by larger cell size and greater shape irregularity. These two biological properties drive the model's predictive power and are consistent with how pathologists diagnose cancer manually under a microscope."*

**Next:** Phase 3 — Model Training in DataRobot